# GGUF Donusumu — VeriYonetim Sorgu Planlayici

Egitim defterinin son hucresi (`save_pretrained_gguf`) Kaggle'da patladi. Sebep disk
degil, **Unsloth'un llama.cpp kurucusunun bozuk olmasi**: llama.cpp Makefile'i kaldirip
CMake'e gecti, Unsloth hala `make clean` cagiriyor; ayrica `LLAMA_CURL is deprecated`
UYARISINI hata sayip iptal ediyor.

Bu defter ayni isi elle yapiyor.

## Onemli: egitim kaybolmadi

Onceki kosuda su ikisi basariyla kaydedildi:

| Klasor | Icerik |
|---|---|
| `lora/` | LoRA eklentisi (~300 MB) — egitimin asil ciktisi |
| `veriyonetim-planlayici/` | **Birlestirilmis 16-bit model** — merge zaten yapildi |

Yani birlestirmeyi tekrarlamiyoruz, dogrudan GGUF'a ceviriyoruz.

## Ayarlar

1. **Add Input → Your Work → hata veren defterini sec → +**
2. **Accelerator = None** (GPU gerekmiyor, kotani yeme)
3. **Internet = On**

## Disk plani

`/kaggle/working` **cikti** icin 20 GB ile sinirli. Ara dosyalari `/tmp`'ye yaziyoruz,
oraya sinir islemiyor. Ciktiya sadece nihai 4,7 GB'lik dosya gidiyor.

| Dosya | Boyut | Nereye |
|---|---|---|
| Birlestirilmis model (girdi) | ~15 GB | `/kaggle/input` (salt okunur) |
| f16 GGUF (ara) | ~15 GB | `/tmp` |
| **q4_k_m GGUF (nihai)** | **~4,7 GB** | `/kaggle/working` |


## 1. Girdiyi bul ve diski kontrol et

In [1]:
import glob, os, subprocess

# Girdi yolu elle yazilmiyor: Kaggle defter ciktisini kendi kurallariyla adlandiriyor.
lora_yollari   = glob.glob("/kaggle/input/**/lora", recursive=True)
merged_yollari = glob.glob("/kaggle/input/**/veriyonetim-planlayici", recursive=True)

if not merged_yollari:
    print("Bulunamadi. /kaggle/input altindaki klasorler:")
    for kok, klasorler, _ in os.walk("/kaggle/input"):
        for k in klasorler:
            print("  ", os.path.join(kok, k))
        break
    raise SystemExit("Girdi bagli mi? Add Input -> Your Work -> defterini sec")

LORA   = lora_yollari[0] if lora_yollari else None
MERGED = merged_yollari[0]

print("LoRA eklentisi     :", LORA)
print("Birlestirilmis model:", MERGED)
print()
print("Birlestirilmis modelin icerigi:")
for d in sorted(os.listdir(MERGED)):
    boyut = os.path.getsize(os.path.join(MERGED, d)) / 1e9
    print(f"  {d:<45} {boyut:6.2f} GB" if boyut > 0.01 else f"  {d}")


LoRA eklentisi     : /kaggle/input/notebooks/muhammetaliyaln/notebooke39056ce43/lora
Birlestirilmis model: /kaggle/input/notebooks/muhammetaliyaln/notebooke39056ce43/veriyonetim-planlayici

Birlestirilmis modelin icerigi:
  .cache
  added_tokens.json
  chat_template.jinja
  config.json
  merges.txt
  model-00001-of-00004.safetensors                4.88 GB
  model-00002-of-00004.safetensors                4.93 GB
  model-00003-of-00004.safetensors                4.33 GB
  model-00004-of-00004.safetensors                1.09 GB
  model.safetensors.index.json
  special_tokens_map.json
  tokenizer.json                                  0.01 GB
  tokenizer_config.json
  vocab.json


In [2]:
# Disk durumu. f16 GGUF ~15 GB yer isteyecek; /tmp'de o kadar bos alan olmali.
!df -h /tmp /kaggle/working


Filesystem      Size  Used Avail Use% Mounted on
overlay         7.9T  6.9T  1.1T  88% /
/dev/loop1       20G   80K   20G   1% /kaggle/working


## 2. LoRA eklentisini Hugging Face'e yedekle

Egitimin asil ciktisi bu ~300 MB'lik klasor. Kaggle defter ciktisi silinebilir; HF'e
koyunca kalici oluyor ve ileride baska bir nicelemeye cevirmek istersen tek satirla
geri geliyor. Yerel internetten tek bayt harcanmiyor.

`HF_TOKEN` Kaggle Secrets'ta tanimli olmali (Add-ons -> Secrets).

In [ ]:
# Kosu basina AYRI depo: yoksa yeni adapter oncekinin ustune yazar.
# Kullanici adi HF hesabiyla ayni olmali (Kaggle kullanici adi ile ayni olmak
# zorunda degil; kosu 1'de burasi bu yuzden 403 vermisti).
HF_DEPO = "rhymali/veriyonetim-planlayici-lora-k2"

# Yedekleme bu defterin ASIL isi degil. Hata firlatirsa "Run All" durur ve
# asil is olan donusum hic calismaz - bu yuzden hata yutuluyor.
if LORA is None:
    print("lora klasoru bulunamadi, bu adim atlandi.")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import HfApi

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

        api = HfApi()
        api.create_repo(HF_DEPO, exist_ok=True, private=True)
        api.upload_folder(folder_path=LORA, repo_id=HF_DEPO)

        print("Yuklendi:", f"https://huggingface.co/{HF_DEPO}")
    except Exception as hata:
        print("YEDEKLEME BASARISIZ (donusum yine de devam edecek):")
        print(" ", type(hata).__name__, hata)
        print("  HF_TOKEN Add-ons -> Secrets'ta tanimli mi? Kullanici adi dogru mu?")
        print("  Onemli: 'lora' klasoru Kaggle ciktisinda duruyor, kaybolmadi.")


## 3. llama.cpp'yi derle

Unsloth'un takildigi yer burasi. Iki fark:

* `make` hic cagrilmiyor — llama.cpp artik yalniz CMake ile derleniyor
* `LLAMA_CURL` bayragi verilmiyor — kaldirilmis, verilince uyari uretiyor

Yalniz `llama-quantize` hedefi derleniyor; sunucu, ornekler ve testler atlaniyor,
boylece derleme dakikalar yerine ~2-3 dakikada bitiyor.

In [ ]:
%%bash
set -e
cd /kaggle/working

if [ ! -d llama.cpp ]; then
    git clone --depth 1 https://github.com/ggml-org/llama.cpp
fi

cd llama.cpp
cmake -B build \
    -DCMAKE_BUILD_TYPE=Release \
    -DGGML_CUDA=OFF \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_EXAMPLES=OFF \
    -DLLAMA_BUILD_SERVER=OFF > /tmp/cmake_yapilandirma.log 2>&1

cmake --build build --config Release -j --target llama-quantize > /tmp/cmake_derleme.log 2>&1

echo "Derleme bitti. llama-quantize konumu:"
find build -name "llama-quantize" -type f


In [ ]:
# Donusturme betiginin bagimliliklari. Kaggle'da torch/transformers/numpy zaten var;
# eksik olan gguf paketi ve tokenizer yardimcilari.
!pip install -q gguf sentencepiece protobuf
print("tamam")


## 4. HF → GGUF (f16)

Cikti `/tmp`'ye yaziliyor — 20 GB'lik cikti sinirina takilmamak icin.

Disk darsa `--outtype f16` yerine `--outtype q8_0` kullanilabilir (~8 GB); nihai
q4_k_m kalitesi cok az duser, ara dosya yariya iner.

In [ ]:
import subprocess, os

komut = [
    "python", "/kaggle/working/llama.cpp/convert_hf_to_gguf.py",
    MERGED,
    "--outfile", "/tmp/veriyonetim-planlayici-f16.gguf",
    "--outtype", "f16",
]
print(" ".join(komut))
print()

sonuc = subprocess.run(komut, capture_output=True, text=True)
print(sonuc.stdout[-3000:])
if sonuc.returncode != 0:
    print("=== HATA ===")
    print(sonuc.stderr[-3000:])
else:
    boyut = os.path.getsize("/tmp/veriyonetim-planlayici-f16.gguf") / 1e9
    print(f"f16 GGUF hazir: {boyut:.1f} GB")


## 5. Niceleme (q4_k_m)

`q4_k_m`, canlida kullanilan `qwen2.5-coder:7b` ile **ayni** niceleme seviyesi — yani
hiz ve bellek davranisi degismiyor, karsilastirma adil kaliyor.

Nihai dosya `/kaggle/working`e yaziliyor; indirilecek tek dosya bu.

In [ ]:
import glob, subprocess, os

quantize = glob.glob("/kaggle/working/llama.cpp/build/**/llama-quantize", recursive=True)[0]
hedef    = "/kaggle/working/veriyonetim-planlayici.Q4_K_M.gguf"

sonuc = subprocess.run(
    [quantize, "/tmp/veriyonetim-planlayici-f16.gguf", hedef, "q4_k_m"],
    capture_output=True, text=True)

print(sonuc.stdout[-2000:])
if sonuc.returncode != 0:
    print("=== HATA ==="); print(sonuc.stderr[-2000:])
else:
    print(f"\nHazir: {os.path.getsize(hedef) / 1e9:.2f} GB")


## 6. Modelfile ve temizlik

In [ ]:
# Model adi "veriyonetim" ile BASLAMALI: sunucu bu onekten modelin ince ayarli
# oldugunu anlayip isteme few-shot ornekleri koymuyor
# (bkz. OllamaOptions.FineTunedPrefix).
#
# temperature/num_ctx burada da veriliyor ama sart degil: QueryPlannerService zaten
# her istekte options.temperature = 0 gonderiyor.
modelfile = """FROM ./veriyonetim-planlayici.Q4_K_M.gguf

PARAMETER temperature 0
PARAMETER num_ctx 4096
"""

with open("/kaggle/working/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

# llama.cpp klasoru cikti sinirini bosuna yiyor, siliniyor.
import shutil
shutil.rmtree("/kaggle/working/llama.cpp", ignore_errors=True)

!ls -lh /kaggle/working


In [ ]:
# GGUF'u HF'e yukle. Yedek amacli DEGIL (GGUF, LoRA'dan 20 dk'da yeniden uretilir);
# amac tasinabilirlik: HF'teki GGUF'u Ollama dogrudan cekiyor, dosya tasimaya gerek
# kalmiyor. Dosya ADI degistirilmemeli - Ollama nicelemeyi ad uzerinden etiketliyor.
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

GGUF_DEPO = "rhymali/veriyonetim-planlayici-gguf-k2"   # kosu basina ayri depo
DOSYA     = "/kaggle/working/veriyonetim-planlayici.Q4_K_M.gguf"

os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))

api = HfApi()
api.create_repo(GGUF_DEPO, exist_ok=True, private=True)
api.upload_file(
    path_or_fileobj = DOSYA,
    path_in_repo    = os.path.basename(DOSYA),
    repo_id         = GGUF_DEPO,
)

print("Yuklendi:", f"https://huggingface.co/{GGUF_DEPO}")
print()
# ollama cp SART: ad "hf.co/..." kalirsa sunucu modeli ince ayarli saymaz, isteme
# 13 few-shot ornegi koyar, model egitimde gormedigi bicimle karsilasir ve dogruluk
# duser - hicbir hata vermeden (bkz. OllamaOptions.FineTunedPrefix).
print("Baska bir makinede kurulum:")
print(f"  ollama pull hf.co/{GGUF_DEPO}:Q4_K_M")
print(f"  ollama cp hf.co/{GGUF_DEPO}:Q4_K_M veriyonetim-planlayici:7b-k2")

## 7. Siradaki adimlar (yerel makinede)

1. `veriyonetim-planlayici.Q4_K_M.gguf` ve `Modelfile` dosyalarini indir, **ayni klasore** koy
2. Modeli Ollama'ya kur:

```bash
ollama create veriyonetim-planlayici:7b -f Modelfile
```

3. Olc ve baz cizgiyle karsilastir:

```bash
cd C:\VeriYonetim
dotnet run --project tools/VeriYonetim.TrainingData -- evaluate ^
    --in data/samples.eval.jsonl --model veriyonetim-planlayici:7b
```

Baz cizgi (qwen2.5-coder:7b, ornekli istem, 300 soru):

| | Ayristi | Gecerli | **Dogru** |
|---|---|---|---|
| eval-seen | %100 | %91,3 | %42,0 |
| eval-unseen | %100 | %96,7 | %49,3 |
| **TOPLAM** | **%100** | **%94,0** | **%45,7** |

Asil bakilacak satir **eval-unseen**: `Filo` ve `Kurs` kataloglari egitime hic girmedi.
Seen ile unseen arasinda buyuk fark acilirsa ezber var demektir.

## Takilirsan

| Belirti | Cozum |
|---|---|
| `numpy` ile ilgili hata | `!pip install -q "numpy<2.0" --force-reinstall` sonra **Run → Restart Session**, 1. hucreden devam |
| `/tmp` doldu | 4. hucrede `--outtype f16` yerine `q8_0` kullan (~8 GB) |
| `llama-quantize` bulunamadi | `/tmp/cmake_derleme.log` dosyasina bak |
| CMake hatasi | `/tmp/cmake_yapilandirma.log` dosyasina bak |
